In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver, gold
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [0]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

In [0]:
try:
    spark
except NameError:
    builder = (
        SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
        .master("local[4]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 1. Set Up and Bronze Data

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

In [0]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [0]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [0]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

# 2 Silver

In [0]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [0]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            df = self.transf_people(df)
        return df

    def transf_people(self, df: DataFrame) -> DataFrame:
        df = (
            df.withColumn("height", df.properties.height)
            .withColumn("mass", df.properties.mass)
            .withColumn("gender", df.properties.gender)
            .drop("url", "properties")
        )
        return df


silver_instance = StarWarsSilver(spark, **options)

In [0]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "people"
)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

# 3 Gold

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [0]:
class StarWarsGold(gold.Gold):
    def people_per_gender(self, df: DataFrame, table: str) -> DataFrame:
        df = df.where("gender <> 'n/a'")
        df = df.where("gender <> 'none'")
        df = df.groupBy("gender").count()
        return df

    def all_females(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("gender = 'female'").drop("LH_SilverTS", "LH_BronzeTS")


gold_instance = StarWarsGold(spark, **options)

In [0]:
gold_instance.load(source_tbl="people").transform(
    tbl_transformations={
        "peoplegender": "people_per_gender",
        "peoplefemale": "all_females",
    }
).write(mode="overwrite", merge_schema=True).execute("peoplegender", "peoplefemale")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplegender")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplefemale")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

# 4 Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")
#spark.stop()